# ЛР1 · 03. Модели для рядов: дообучение (few-shot) и LoRA

Порядок по важности (при сбое главное уже сохранено):
1. **Chronos-2 + LoRA** — параметрически эффективная адаптация (PEFT) на последних 5% train;
2. **Chronos-2, полное дообучение** на тех же данных (few-shot);
3. **TTM r2, полное дообучение** на тех же данных (few-shot).

«Последние 5% train» = 59 дней до 31.03.2026, все 424 станции (PROTOCOL.md, п. 4). Тест и подвыборка — те же.
Zero-shot всех шести моделей — в `lab-01`. Chronos-Bolt, TimesFM, Moirai, Lag-Llama: дообучение в их
официальных библиотеках недоступно без отдельной обвязки (AutoGluon / собственный цикл обучения) — в отчёте «н/д».

**Kaggle:** Inputs — датасет и вывод `lab-00-setup`; Accelerator — **GPU T4 x2**; Internet — **On**.

## 0. Установка

In [ ]:
import subprocess, sys, os
def pip(*args):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], capture_output=True, text=True)
    print("pip install", *args, "->", "OK" if r.returncode == 0 else "ОШИБКА\n" + r.stderr[-1500:])
pip("chronos-forecasting", "peft")
pip("--no-deps", "granite-tsfm")

In [ ]:
import glob, json, os, sys, time, traceback, gc, importlib.metadata
from pathlib import Path
import numpy as np, pandas as pd, torch

# labkit и подвыборка — из вывода ноутбука lab-00-setup
hits = glob.glob("/kaggle/input/**/labkit.py", recursive=True) or glob.glob("labkit.py")
LAB00 = Path(hits[0]).parent
sys.path.insert(0, str(LAB00))
import labkit as lk
print("labkit:", LAB00)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE,
      torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")
for p in ["chronos-forecasting", "peft", "granite-tsfm", "transformers"]:
    try:
        print(f"  {p}: {importlib.metadata.version(p)}")
    except Exception:
        print(f"  {p}: не установлен")

## 1. Данные и окна

In [ ]:
d = lk.load()
sub = json.load(open(LAB00 / "subset.json", encoding="utf-8"))
stations = sub["stations"]
test_orig = [pd.Timestamp(x) for x in sub["test_origins"]]
val_orig = [pd.Timestamp(x) for x in sub["val_origins"]]
assert stations == lk.make_subset(d), "подвыборка не совпадает с lab-00!"
tmax = d.train_max()

W = {("val", 336): lk.make_windows(d, stations, val_orig, context=336)}
for c in lk.CONTEXTS:
    W[("test", c)] = lk.make_windows(d, stations, test_orig, context=c)
for k, w in W.items():
    print(k, len(w), "окон, контекст", w.ctx.shape[1])
Q = list(lk.QUANTILES)

## 2. Общий запуск модели (как в lab-01)

In [ ]:
ERRORS = lk.results_dir() / "errors.csv"

def log_error(model, stage, exc):
    tb = traceback.format_exc()
    print(f"!!! {model} [{stage}] не отработал: {exc}\n{tb[-1500:]}")
    pd.DataFrame([{"date": pd.Timestamp.now(), "model": model, "stage": stage,
                   "error": repr(exc)[:500], "traceback": tb[-3000:]}]).to_csv(
        ERRORS, mode="a", header=not ERRORS.exists(), index=False)

def evaluate(model, mode, w, predict, batch=128, extra=None):
    """predict(ctx_batch) -> (point [b,24], quant [b,24,9] или None). Возвращает строку runs.csv."""
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    t = lk.Timer()
    pts, qs = [], []
    for i in range(0, len(w), batch):
        with t:
            p, q = predict(w.ctx[i:i + batch], i)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
        pts.append(np.asarray(p, dtype="float64"))
        qs.append(None if q is None else np.asarray(q, dtype="float64"))
    point = np.concatenate(pts)
    quant = None if qs[0] is None else np.concatenate(qs)
    # Постобработка для счётных данных: отрицательные значения обрезаются до 0 (PROTOCOL.md, п. 5)
    point = np.clip(point, 0, None)
    if quant is not None:
        quant = np.clip(np.sort(quant, axis=-1), 0, None)
    valid = lk.check_valid(list(point), w, tmax)
    point = lk.apply_fallback(list(point), valid, w)
    if quant is not None:
        quant[~valid] = point[~valid][..., None]
    timing = {}
    times = np.array(t.times) * 1000
    sizes = [min(batch, len(w) - i) for i in range(0, len(w), batch)]
    per = times / np.array(sizes)
    timing = {"ms_p50": float(np.median(per)), "ms_p95": float(np.quantile(per, 0.95)), "total_s": float(times.sum() / 1000)}
    row = lk.save_run(model, mode, w, point, valid, quant, timing,
                      {"context": w.ctx.shape[1], **(extra or {})})
    print(f"  {model:14s} {mode:22s} WAPE={row['wape']:.4f} [{row['wape_ci_lo']:.4f}; {row['wape_ci_hi']:.4f}] "
          f"невалидных={row['invalid_share']:.1%} p95={row['ms_p95']:.1f} мс/окно GPU={row['gpu_peak_gb']:.2f} ГБ")
    return row

def run_all(model, make_predict, extra=None):
    """make_predict(context_length) -> predict. Сначала val (контекст 336), затем тест на трёх контекстах."""
    plan = [("val", 336)] + [("test", c) for c in lk.CONTEXTS]
    for part, c in plan:
        try:
            pred = make_predict(c)
            evaluate(model, f"zs_ctx{c}_{part}", W[(part, c)], pred, extra=extra)
        except Exception as e:
            log_error(model, f"zs_ctx{c}_{part}", e)
            if part == "val":
                print(f"  {model}: на val не отработал — тест пропускается")
                return
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 3. Данные для дообучения: последние 5% train

59 дней до конца train (≈ 5% от 28 460 часов), все 424 станции. Для Chronos-2 — ряд каждой станции
длиной 59 дней + 672 ч истории (закрытые часы → NaN, модель умеет пропуски). Для TTM — окна
«512 ч → 24 ч» с точкой прогноза в 00:00 каждого из 59 дней (окна с закрытиями в горизонте пропускаются).

In [ ]:
FT_DAYS = 59
idx = d.y.index
cols = list(d.y.columns)
Yall = d.y.to_numpy(dtype="float64")
train_end = d.part("train")[1]
p_end = idx.get_loc(train_end) + 1                      # позиция первого часа val
p_ft0 = idx.get_loc(train_end.normalize() - pd.Timedelta(days=FT_DAYS - 1))   # 00:00 первого дня дообучения
Ymask = np.where(d.closed.to_numpy(), np.nan, Yall)
ft_series = [Ymask[p_ft0 - 672:p_end, k].astype("float32") for k in range(len(cols))]
print(f"Дообучение: {idx[p_ft0]:%d.%m.%Y} – {idx[p_end - 1]:%d.%m.%Y %H:%M}, рядов {len(ft_series)}, "
      f"длина {len(ft_series[0])}")

## 4. Chronos-2: LoRA и полное дообучение

In [ ]:
os.environ["WANDB_DISABLED"] = "true"
from chronos import Chronos2Pipeline
base = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map=DEVICE)

def c2_predict_fn(pipe):
    def make(c):
        def predict(ctx, i):
            q, _ = pipe.predict_quantiles(ctx[:, None, :].astype("float32"), prediction_length=lk.HORIZON,
                                          quantile_levels=Q, batch_size=128)
            q = np.stack([x[0].cpu().numpy() for x in q])
            return q[..., Q.index(0.5)], q
        return predict
    return make

FT_RUNS = [("lora", dict(finetune_mode="lora", learning_rate=1e-4)),
           ("full", dict(finetune_mode="full", learning_rate=1e-5))]
for name, kw in FT_RUNS:
    try:
        t0 = time.time()
        torch.cuda.reset_peak_memory_stats()
        ft = base.fit(ft_series, prediction_length=lk.HORIZON, context_length=672, num_steps=500, batch_size=64,
                      output_dir=f"/tmp/c2_{name}", **kw)
        train_min = (time.time() - t0) / 60
        train_gb = torch.cuda.max_memory_allocated() / 1e9
        print(f"Chronos-2 {name}: обучение {train_min:.1f} мин, пик памяти {train_gb:.2f} ГБ")
        extra = {"weights": "amazon/chronos-2", "finetune": name, "ft_steps": 500, "ft_lr": kw["learning_rate"],
                 "ft_minutes": round(train_min, 2), "ft_gpu_gb": round(train_gb, 2)}
        for c in [336, 672]:
            evaluate("chronos2", f"{'peft_lora' if name == 'lora' else 'fs5pct'}_ctx{c}_test", W[("test", c)],
                     c2_predict_fn(ft)(c), extra=extra)
        del ft
    except Exception as e:
        log_error("chronos2", f"finetune_{name}", e)
    gc.collect(); torch.cuda.empty_cache()
del base

## 5. TTM r2: дообучение

In [ ]:
from tsfm_public.toolkit.get_model import get_model

def load_ttm(c):
    want = 512 if c <= 512 else 1024
    m = get_model("ibm-granite/granite-timeseries-ttm-r2", context_length=want, prediction_length=lk.HORIZON)
    L = m.config.context_length
    print(f"  TTM: контекст данных {c} ч, окно модели {L} ч")
    return m.to(DEVICE), L

def ttm_inputs(ctx, L):
    c = ctx.shape[1]
    x = np.zeros((len(ctx), L), dtype="float32"); m = np.zeros_like(x)
    k = min(c, L)
    x[:, -k:] = ctx[:, -k:]; m[:, -k:] = 1                 # слева нули, помеченные как «не наблюдалось»
    return torch.tensor(x[..., None], device=DEVICE), torch.tensor(m[..., None], device=DEVICE)

def ttm_predict_fn(model, L):
    def predict(ctx, i):
        model.eval()
        x, m = ttm_inputs(ctx, L)
        with torch.no_grad():
            out = model(past_values=x, past_observed_mask=m)
        return out.prediction_outputs[:, :lk.HORIZON, 0].float().cpu().numpy(), None
    return predict

In [ ]:
# Дообучение TTM: окна «512 ч → 24 ч», точки прогноза в 00:00 каждого из 59 дней, все станции
try:
    model, L = load_ttm(336)
    X, M, Yt = [], [], []
    for day in pd.date_range(train_end.normalize() - pd.Timedelta(days=FT_DAYS - 1), train_end.normalize()):
        p = idx.get_loc(day)                                # 00:00 каждого дня (с учётом перевода часов)
        for k in range(len(cols)):
            fut = Ymask[p:p + lk.HORIZON, k]
            hist = Ymask[p - L:p, k]
            if np.isnan(fut).any():
                continue
            X.append(np.nan_to_num(hist)); M.append(~np.isnan(hist)); Yt.append(fut)
    X, M, Yt = (np.array(a, dtype="float32") for a in (X, M, Yt))
    print(f"TTM: окон для дообучения {len(X):,}")
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
    rng = np.random.default_rng(lk.SEED)
    torch.cuda.reset_peak_memory_stats(); t0 = time.time()
    model.train()
    for epoch in range(2):
        order = rng.permutation(len(X)); losses = []
        for b in range(0, len(X), 64):
            j = order[b:b + 64]
            x = torch.tensor(X[j][..., None], device=DEVICE); m = torch.tensor(M[j][..., None].astype("float32"), device=DEVICE)
            y = torch.tensor(Yt[j], device=DEVICE)
            pred = model(past_values=x, past_observed_mask=m).prediction_outputs[:, :lk.HORIZON, 0]
            scale = (x[..., 0] * m[..., 0]).sum(1, keepdim=True) / m[..., 0].sum(1, keepdim=True).clamp(min=1) + 1
            loss = ((pred - y).abs() / scale).mean()            # MAE в масштабе станции
            opt.zero_grad(); loss.backward(); opt.step(); losses.append(loss.item())
        print(f"  эпоха {epoch + 1}: средняя ошибка {np.mean(losses):.4f}")
    train_min = (time.time() - t0) / 60
    extra = {"weights": "ibm-granite/granite-timeseries-ttm-r2", "finetune": "full", "ft_epochs": 2, "ft_lr": 1e-4,
             "ft_minutes": round(train_min, 2), "ft_gpu_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2),
             "model_context": L}
    evaluate("ttm_r2", "fs5pct_ctx336_test", W[("test", 336)], ttm_predict_fn(model, L), extra=extra)
    del model
except Exception as e:
    log_error("ttm_r2", "finetune_full", e)
gc.collect(); torch.cuda.empty_cache()

## 6. Сводка

In [ ]:
runs = pd.read_csv(lk.results_dir() / "runs.csv")
show = ["model", "mode", "invalid_share", "wape", "wape_ci_lo", "wape_ci_hi", "mase", "wql", "cov80",
        "ms_p95", "gpu_peak_gb", "ft_minutes"]
pd.set_option("display.width", 220)
print(runs[[c for c in show if c in runs]].round(4).to_string())
print("\nОшибки:\n" + (pd.read_csv(ERRORS)[["model", "stage", "error"]].to_string() if ERRORS.exists() else "нет"))